In [ ]:
source_catalog = spark.conf.get("source_catalog")
source_schema = spark.conf.get("source_schema")
table_prefix = spark.conf.get("table_prefix")
source_table = f"{source_catalog}.{source_schema}.{table_prefix}_otel_logs"

In [ ]:
@dlt.table(
    comment="All OTel logs with flattened attributes and severity parsed from body text",
)
def logs_flattened():
    return (
        spark.readStream.table(source_table)
        .filter(F.col("time_unix_nano").isNotNull())
        .selectExpr(
            "TIMESTAMP_MICROS(CAST(time_unix_nano / 1000 AS BIGINT)) AS log_timestamp",
            "TIMESTAMP_MICROS(CAST(observed_time_unix_nano / 1000 AS BIGINT)) AS observed_timestamp",
            "ROUND((observed_time_unix_nano - time_unix_nano) / 1e6, 2) AS ingestion_lag_ms",
            SEVERITY_EXPR,
            "body",
            "LENGTH(body) AS body_length",
            "CASE WHEN body LIKE '\\tat %' OR body LIKE 'Caused by:%' THEN TRUE ELSE FALSE END AS is_stack_trace_line",
            # K8s resource hierarchy
            "resource.attributes['k8s.node.name'] AS node",
            "resource.attributes['k8s.namespace.name'] AS namespace",
            "resource.attributes['k8s.deployment.name'] AS deployment",
            "resource.attributes['k8s.daemonset.name'] AS daemonset",
            "resource.attributes['k8s.pod.name'] AS pod",
            "resource.attributes['k8s.pod.uid'] AS pod_uid",
            "resource.attributes['k8s.container.name'] AS container",
            "CAST(resource.attributes['k8s.container.restart_count'] AS INT) AS container_restart_count",
            "resource.attributes['k8s.replicaset.name'] AS replicaset",
            "resource.attributes['k8s.cluster.uid'] AS cluster_uid",
            # Service identity
            "resource.attributes['service.name'] AS service",
            "resource.attributes['service.namespace'] AS service_namespace",
            "resource.attributes['service.instance.id'] AS service_instance_id",
            "resource.attributes['service.version'] AS service_version",
            "resource.attributes['container.image.name'] AS image_name",
            "resource.attributes['container.image.tag'] AS image_tag",
            # Log-level attributes
            "attributes['log.file.path'] AS log_file_path",
            "attributes['log.iostream'] AS log_iostream",
            "attributes['logtag'] AS logtag",
            # Trace correlation
            "NULLIF(trace_id, '') AS trace_id",
            "NULLIF(span_id, '') AS span_id",
        )
    )

In [ ]:
@dlt.table(
    comment="Error and warning logs for monitoring and alerting",
)
def logs_errors():
    return (
        dlt.read_stream("logs_flattened")
        .filter(F.col("parsed_severity").isin("ERROR", "WARN", "FATAL"))
        .select(
            "log_timestamp",
            "observed_timestamp",
            "ingestion_lag_ms",
            "parsed_severity",
            "node",
            "namespace",
            "deployment",
            "daemonset",
            "pod",
            "container",
            "container_restart_count",
            "service",
            "service_namespace",
            "image_name",
            "image_tag",
            "body",
            "body_length",
            "is_stack_trace_line",
            "log_file_path",
            "log_iostream",
            "trace_id",
            "span_id",
        )
    )